# vLLM Quickstart 한국어 튜토리얼

공식 vLLM 저장소(`vllm-project/vllm`) 태그 **v0.30.0** 기준 공식 예제를 번역+해설한 튜토리얼입니다.
원본: `examples/basic/offline_inference/basic.py`, `examples/basic/online_serving/openai_chat_completion_client.py`

**권장 런타임**: GPU(L4 권장, T4는 최근 알려진 버그로 불안정함). 메뉴 → 런타임 유형 변경 → 하드웨어 가속기(L4).

## 이 튜토리얼에서 배우는 것
1. offline 추론(`LLM` 클래스로 바로 프롬프트 넣고 결과 받기)
2. OpenAI 호환 서버 기동하고 API로 호출하기
3. 시작 로그에서 **어떤 attention 백엔드가 선택되는지** 확인하고, 왜 그 백엔드가 뜨는지 코드 근거로 설명

## 유의점 (미리 읽어두면 사고가 안 생김)
- **offline과 서버를 같은 커널에서 연속 실행하면 GPU 메모리가 안 풀려 OOM이 납니다.** 이 노트북은 그래서 offline 파트와 서버 파트를 **서로 다른 런타임**으로 나눠 실행하도록 구성했습니다(또는 서버 셀 실행 전 커널 재시작 필요).
- 서버는 백그라운드로 띄우고 `/health`를 폴링해야 합니다. 포그라운드로 띄우면 셀 실행이 끝나지 않습니다.
- **`pip install vllm`이 torch를 새 버전으로 끌어올리면서 Colab 기본 torchaudio와 CUDA 버전이 어긋나 `RuntimeError`가 날 수 있습니다.** 설치 셀에서 torchaudio를 함께 제거해 회피합니다(텍스트 전용 튜토리얼이라 불필요).
- **Jupyter 안에서 `LLM(...)`을 직접 호출하면 `io.UnsupportedOperation: fileno`로 죽을 수 있습니다.** vLLM 워커 프로세스가 기본값(`fork`)으로 뜨면서 Jupyter의 특수한 stdout 객체를 물려받아 생기는 문제 — `VLLM_WORKER_MULTIPROC_METHOD=spawn`으로 회피합니다(`vllm serve` CLI로 띄우는 서버는 별도 셸 프로세스라 이 문제가 없습니다).
- 실습이 끝나면 **런타임을 바로 종료**하세요 — L4는 켜져 있는 동안 계속 Colab 컴퓨트 유닛을 소진합니다.


## 0. 환경 확인
재현성을 위해 먼저 GPU/드라이버 정보를 남깁니다.

In [5]:
!nvidia-smi
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"

Sun Sep 27 06:20:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P0             29W /   72W |   20810MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. vLLM 설치 (v0.30.0 고정)
이 튜토리얼은 버전 변화에 흔들리지 않도록 태그를 고정합니다.

In [6]:
!pip install -q vllm==0.30.0 openai
# transformers가 import 시 torchaudio를 함께 로드하는데, vllm이 끌어올린 torch(CUDA 13.0)와
# Colab 기본 torchaudio(CUDA 12.8)의 버전이 어긋나 RuntimeError가 납니다.
# 이 튜토리얼은 텍스트 전용이라 torchaudio 자체가 불필요 — 제거해서 충돌을 피합니다.
!pip uninstall -q -y torchaudio

## 2. Offline 추론
원본: `examples/basic/offline_inference/basic.py` (v0.30.0)

`LLM` 클래스가 모델을 로드하고, `generate()`가 프롬프트 리스트를 받아 한 번에 배치 실행합니다.

In [7]:
import os

# vLLM은 기본적으로 워커 프로세스를 fork로 띄우는데, fork는 부모(Jupyter 커널)의 stdout을
# 그대로 물려받습니다. Jupyter의 stdout은 진짜 OS 파일 디스크립터가 없는 특수 객체라
# vLLM 내부의 `sys.stdout.fileno()` 호출에서 `io.UnsupportedOperation: fileno`로 죽습니다.
# spawn으로 바꾸면 자식 프로세스가 새 인터프리터로 시작해 이 문제를 피합니다.
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# attention 백엔드 자동 선택 결과는 기본(INFO) 로그 레벨엔 안 찍히고 DEBUG에만 남습니다
# (다음 마크다운 셀 참고) — 그래서 처음부터 DEBUG로 띄웁니다.
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

from vllm import LLM, SamplingParams

# 예제 프롬프트
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
# 샘플링 파라미터 객체
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)

llm = LLM(model="facebook/opt-125m")

outputs = llm.generate(prompts, sampling_params)

print("\n생성 결과:\n" + "-" * 60)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt:    {prompt!r}")
    print(f"Output:    {generated_text!r}")
    print("-" * 60)

INFO 09-27 06:20:51 [api_utils.py:286] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 09-27 06:20:52 [model.py:692] Resolved architecture: OPTForCausalLM
INFO 09-27 06:20:52 [model.py:2030] Using max model len 2048
INFO 09-27 06:20:52 [kernel.py:416] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
INFO 09-27 06:21:08 [utils.py:632] [shutdown] Process manager: send sigterm to process EngineCore


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

### 어떤 attention 백엔드가 선택되었나? (정정)
공식 문서/여러 블로그는 "시작 로그에 `Using X attention backend`가 찍힌다"고 설명하지만,
v0.30.0 소스(`platforms/cuda.py`)를 직접 보면 그 `logger.info("Using %s backend.", ...)` 줄은
**`--attention-backend`로 백엔드를 직접 지정했을 때만** 실행됩니다. 우리처럼 아무것도 지정하지
않고 자동 선택에 맡기면 결과는 `logger.debug_once(...)`로만 남습니다 — 그래서 위 셀을
`VLLM_LOGGING_LEVEL=DEBUG`로 띄웠습니다.

위 로그를 `DEBUG` 문자열로 검색해 `Some attention backends are not valid for ...`로 시작하는 줄을
찾으세요. 그 줄의 `Reasons: {...}` 안에 **탈락한 backend와 이유만** 나열됩니다 — 선택된 backend는
그 목록에 없는 것으로 역으로 추론해야 합니다(vLLM이 "이게 선택됨"이라고 INFO로 따로 찍어주지
않기 때문). 선택 우선순위 자체는 `vllm/platforms/cuda.py`의 `_get_backend_priorities()`에 있고,
대체로 FLASH_ATTN/FLASHINFER가 우선 시도되고, 그 둘 중 현재 설정을 지원하지 못하면 TRITON_ATTN
으로 폴백합니다. L4(SM89)는 bf16/fp8까지 지원하는 비교적 새 아키텍처라 대부분 FLASH_ATTN이
선택됩니다.

## 3. 커널 재시작 (중요)

**유의점**: 위에서 만든 `llm` 객체가 GPU 메모리를 점유한 채로 남아있습니다. 이 상태에서 서버를 또 띄우면 OOM이 납니다.
**Colab 메뉴에서 `런타임 > 셀 실행 후 다시 시작`(Restart runtime)을 실행한 다음**, 아래 셀부터 이어서 진행하세요.
(nbconvert 자동실행 시에는 이 노트북을 두 개로 분리해 실행하는 것을 권장합니다.)

## 4. OpenAI 호환 서버 기동
원본: `examples/basic/online_serving/openai_chat_completion_client.py` (v0.30.0)

서버는 백그라운드 프로세스로 띄웁니다. 포그라운드로 띄우면 셀 실행이 영원히 끝나지 않습니다.

In [ ]:
import subprocess
import time
import urllib.request

log_path = "/content/vllm_server.log"
log_file = open(log_path, "w")

server_proc = subprocess.Popen(
    ["vllm", "serve", "facebook/opt-125m"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print(f"서버 PID: {server_proc.pid}, 로그: {log_path}")

# /health를 폴링해서 서버가 뜨기까지 기다립니다(모델 로드 포함 최대 몇 분 걸릴 수 있음).
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("서버 준비 완료")
        break
    except Exception:
        time.sleep(5)
else:
    print("타임아웃 — 로그를 확인하세요:")
    print(open(log_path).read()[-2000:])

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

models = client.models.list()
model = models.data[0].id

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who won the world series in 2020?"},
]

chat_completion = client.chat.completions.create(messages=messages, model=model)
print(chat_completion)

## 5. 종료 (반드시 실행)
자원(GPU/컴퓨트 유닛)을 아끼기 위해 서버 프로세스를 종료하고, 이 셀을 마지막으로 실행한 후 런타임도 종료하세요.

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=30)
print("서버 종료됨")